In [ ]:
# nutrition_estimator.py

import json
import difflib
import pandas as pd

# Load nutrition data
nutrition_df = pd.read_csv("/content/sample_data/Assignment Inputs - Nutrition source.csv")
nutrition_df = nutrition_df[[
    "food_name", "energy_kcal", "protein_g", "carb_g", "fat_g", "fibre_g"
]].dropna(subset=["food_name"])

In [ ]:
# Estimated household measurement to grams (rough estimates)
MEASUREMENT_TO_GRAMS = {
    "cup cubes": 120,
    "teaspoons": 5,
    "tablespoon": 15,
    "cup puree": 100,
    "cup chopped": 100,
}

# Simulated recipe fetch
RECIPES = {
    "Paneer Butter Masala": [
        {"ingredient": "Paneer", "quantity": "0.75 cup cubes"},
        {"ingredient": "Butter", "quantity": "2 teaspoons"},
        {"ingredient": "Tomato", "quantity": "0.5 cup puree"},
        {"ingredient": "Onion", "quantity": "0.5 cup chopped"},
        {"ingredient": "Cream", "quantity": "1 tablespoon"}
    ]
}

In [ ]:
# Dish classification
DISH_TYPES = {
    "Paneer Butter Masala": "Wet Sabzi"
}

DISH_TYPE_GRAMS = {
    "Wet Sabzi": 180
}

def parse_quantity(qty_str):
    try:
        parts = qty_str.split()
        value = float(parts[0])
        unit = " ".join(parts[1:])
        return value * MEASUREMENT_TO_GRAMS.get(unit, 0)
    except:
        return 0

def match_ingredient(name):
    all_ingredients = nutrition_df["food_name"].tolist()
    match = difflib.get_close_matches(name, all_ingredients, n=1, cutoff=0.6)
    return match[0] if match else None

def estimate_nutrition(dish_name):
    ingredients = RECIPES.get(dish_name, [])
    total_nutrition = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0}
    total_weight = 0

    used_ingredients = []

    for item in ingredients:
        ing = item["ingredient"]
        qty_str = item["quantity"]
        weight_g = parse_quantity(qty_str)
        total_weight += weight_g

        matched_ing = match_ingredient(ing)
        if matched_ing:
            row = nutrition_df[nutrition_df["food_name"] == matched_ing].iloc[0]
            total_nutrition["calories"] += (weight_g / 100) * row["energy_kcal"]
            total_nutrition["protein"] += (weight_g / 100) * row["protein_g"]
            total_nutrition["carbs"] += (weight_g / 100) * row["carb_g"]
            total_nutrition["fat"] += (weight_g / 100) * row["fat_g"]

        used_ingredients.append({"ingredient": ing, "quantity": qty_str})

    dish_type = DISH_TYPES.get(dish_name, "Unknown")
    serving_weight = DISH_TYPE_GRAMS.get(dish_type, 100)
    scaling_factor = serving_weight / total_weight if total_weight else 0

    scaled_nutrition = {
        k: round(v * scaling_factor) for k, v in total_nutrition.items()
    }

    return {
        "estimated_nutrition_per_200ml_katori": scaled_nutrition,
        "dish_type": dish_type,
        "ingredients_used": used_ingredients
    }

if __name__ == "__main__":
    dish_input = input("Enter dish name: ")
    output = estimate_nutrition(dish_input)
    print(json.dumps(output, indent=2))


Enter dish name: Paneer Butter Masala
{
  "estimated_nutrition_per_200ml_katori": {
    "calories": 336,
    "protein": 24,
    "carbs": 14,
    "fat": 20
  },
  "dish_type": "Wet Sabzi",
  "ingredients_used": [
    {
      "ingredient": "Paneer",
      "quantity": "0.75 cup cubes"
    },
    {
      "ingredient": "Butter",
      "quantity": "2 teaspoons"
    },
    {
      "ingredient": "Tomato",
      "quantity": "0.5 cup puree"
    },
    {
      "ingredient": "Onion",
      "quantity": "0.5 cup chopped"
    },
    {
      "ingredient": "Cream",
      "quantity": "1 tablespoon"
    }
  ]
}
